# 5. Machine learning: predict first-stage landing success
This notebook evaluates models using a train/test split. Metrics are computed from the dataset actually loaded; do not copy unverified scores into the final presentation.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
DATA = ROOT / "data"
file = next((DATA / n for n in ["dataset_part_3.csv", "dataset_part_2.csv", "dataset_part_1.csv"] if (DATA/n).exists()), None)
if file is None: raise FileNotFoundError("Run src/download_data.py first.")
df = pd.read_csv(file)
print("Using:", file.name, "| shape:", df.shape)
display(df.head())

In [ ]:
target = next((c for c in ["Class", "class", "landing_success", "LandingSuccess"] if c in df.columns), None)
if target is None:
    raise ValueError(f"Could not identify binary target. Columns are: {df.columns.tolist()}")
y = df[target]
X = df.drop(columns=[target])
# Remove columns that are likely identifiers or raw text notes, if present.
id_like = [c for c in X.columns if c.lower() in {"flightnumber", "flight_number", "date", "serial"}]
X = X.drop(columns=id_like, errors="ignore")
# Remove rows with missing target.
valid = y.notna()
X, y = X.loc[valid].copy(), y.loc[valid].copy()
if y.nunique() != 2:
    raise ValueError(f"Expected binary target but found {y.nunique()} unique values: {y.unique()}")
print("Target distribution:")
display(y.value_counts(dropna=False))

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                    "Precision": precision_score(y_test, pred, average="binary", zero_division=0),
                    "Recall": recall_score(y_test, pred, average="binary", zero_division=0),
                    "F1": f1_score(y_test, pred, average="binary", zero_division=0)})
results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df)

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
print("Best model by F1:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred)
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## Interpretation checklist
- Compare precision, recall, and F1, not accuracy alone.
- Confirm which class represents successful landing before interpreting metrics.
- The test set is small, so metrics may vary with the split and dataset version.
- Report the measured scores from your own notebook execution.